In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("OpenAI API key is configured")
else:
    print("OpenAI API key is not set")

In [ ]:
openai = OpenAI()
MODEL = 'gpt-5-nano'

In [ ]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
OLLAMA_MODEL = 'llama3.2'

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
system_message = 'You are a support agent for an airline. You answer user questions. You also have ability to query for ticket prices, just respond only "Use tool to fetch ticket price for London" to retrieve the ticket price for London, or for a city you name.'

In [ ]:
message = "I'd like to go to Paris. How much is the flight?"

In [ ]:
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": message}]
response = openai.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)


In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
history = []
message = "I'd like to go to Paris. How much is the flight?"
chat(message, history)



In [ ]:
system_message = ""
print(chat(message, history))

In [ ]:
message = """
Find current flight fares to Paris, France (CDG and/or ORY). Departure airport: SFO. Trip: Round trip. Departure date: 2026-10-25. Return date: 2026-11-15. Date flexibility: ±3 days. Travelers: 1 adult. Cabin: Economy. Stops: Nonstop preferred. Baggage: Carry-on.
"""
print(chat(message, history))


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [ ]:
get_ticket_price("Berlin")

In [ ]:
get_ticket_price("London")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the stored ticket price for a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

set_ticket_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the stored ticket price in USD for a city. Use only when explicitly asked to set or change a price.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Destination city whose ticket price should be set",
            },
            "price": {
                "type": "number",
                "description": "New ticket price in US dollars; must be non-negative",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_ticket_price_function},
]

In [ ]:
tools

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
response = openai.responses.create(
    model="gpt-5-nano",
    tools=[{"type": "web_search"}],
    input="Find current round-trip economy fares from SFO to Paris for October 25–November 15, 2026. Cite sources.",
)

print(response.output_text)

In [ ]:
usage = response.usage

print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

print("Input details:", usage.input_tokens_details)
print("Output details:", usage.output_tokens_details)

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
!ollama pull qwen3.5:27b


In [ ]:
!ollama list


In [ ]:
def ollama_chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen3.5:27b", messages=messages)
    return response.choices[0].message.content

In [ ]:
message = "Please quick sort in python"
history = []
ollama_chat(message, history)



In [ ]:
system_message = ""

In [ ]:
message = "Please write a quick sort function in python"
print(ollama_chat(message, history))

In [ ]:
message = "Please write a quick sort function in java"
print(ollama_chat(message, history))

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    print(f"messages: {messages}")
    print(f"response: {response}")

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        for message in messages:
            print(f"inside - message: {message}")
        print(f"inside - response: {response}")
    
    return response.choices[0].message.content

In [ ]:
import json

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_ticket_price":
            city = arguments["destination_city"]
            result = get_ticket_price(city)
        elif tool_call.function.name == "set_ticket_price":
            city = arguments["city"]
            price = arguments["price"]
            result = set_ticket_price(city, price)
        else:
            result = f"Unsupported tool: {tool_call.function.name}"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id,
        })
        print(f"Tool {tool_call.function.name} arguments: {arguments}")
        print(f"Tool result: {result}")

    return responses

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [ ]:
!ollama list


In [ ]:
OLLAMA_MODEL = "qwen3.5:27b"

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages, tools=tools)
    print(f"messages: {messages}")
    print(f"response: {response}")

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=messages)
        for message in messages:
            print(f"inside - message: {message}")
        print(f"inside - response: {response}")
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=ollama_chat).launch()

In [ ]:
import sqlite3

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
def set_ticket_price(city, price):
    city = city.strip().lower()
    price = float(price)
    if not city:
        raise ValueError("City must not be empty")
    if price < 0:
        raise ValueError("Ticket price must be non-negative")

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) "
            "ON CONFLICT(city) DO UPDATE SET price = excluded.price",
            (city, price),
        )
        conn.commit()

    return f"Updated the ticket price for {city.title()} to ${price:.2f}."

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
set_ticket_price("Tokyo", 1000)

In [ ]:
gr.ChatInterface(fn=chat).launch()